# Exploration des données foudre — Meteorage
Analyse du fichier `segment_alerts_all_airports_train.csv`

In [ ]:
import pandas as pd

## 1. Import des données

In [ ]:
FILE_PATH = "segment_alerts_all_airports_train.csv"

df = pd.read_csv(FILE_PATH)
print("Fichier chargé avec succès")

## 2. Aperçu général

In [ ]:
print(f"Nombre de lignes    : {len(df):,}")
print(f"Nombre de colonnes  : {df.shape[1]}")
print(f"\nColonnes : {list(df.columns)}")

In [ ]:
# type des colonnes

df.dtypes

In [ ]:
# conversion de "date" en type "datetime"
df['date'] = pd.to_datetime(df['date'], utc=True)

In [ ]:
# les 5 premières valeurs

df.head()

## 3. Valeurs manquantes

In [ ]:
missing = pd.DataFrame({
    'nb_manquants': df.isnull().sum(),
    'pct_manquants (%)': (df.isnull().sum() / len(df) * 100).round(2)
})

missing = missing.sort_values('pct_manquants (%)', ascending=False)
print(missing.to_string())

## 4. Statistiques descriptives

In [ ]:
df.describe(include='all')

In [ ]:
# nombre d'éclairs en terme de distance

bins = [0, 20, 30, float('inf')]
labels = ['< 20 km', '20-30 km', '> 30 km']

dist_cat = pd.cut(df['dist'], bins=bins, labels=labels)

result = dist_cat.value_counts().sort_index().to_frame(name='nb_eclairs')
result['pct (%)'] = (result['nb_eclairs'] / len(df) * 100).round(2)

result

In [ ]:
# Pourcentage par type d'éclair
print("=== Type d'éclair ===")
type_result = df['icloud'].value_counts().to_frame(name='nb_eclairs')
type_result.index = ['Intra-nuage (True)', 'Nuage-sol (False)']
type_result['pct (%)'] = (type_result['nb_eclairs'] / len(df) * 100).round(2)
print(type_result)

In [ ]:
# Croisement distance x type
print("\n=== Croisement distance x type ===")
bins = [0, 20, 30, float('inf')]
labels = ['< 20 km', '20-30 km', '> 30 km']

dist_cat = pd.cut(df['dist'], bins=bins, labels=labels)

cross = pd.crosstab(dist_cat, df['icloud'], margins=True)
cross.columns = ['Nuage-sol', 'Intra-nuage', 'Total']
cross['pct_nuage_sol (%)'] = (cross['Nuage-sol'] / cross['Total'] * 100).round(2)
cross['pct_intra_nuage (%)'] = (cross['Intra-nuage'] / cross['Total'] * 100).round(2)

cross

In [46]:
# stats sur la durée d'une alerte en minutes

# Filtrer les éclairs avec un alert_airport_id
alerts = df[df['airport_alert_id'].notna()].copy()

# Durée de chaque alerte = max(date) - min(date) par alerte
duree = alerts.groupby(['airport', 'airport_alert_id'])['date'].agg(
    debut='min',
    fin='max'
)

duree['duree_minutes'] = (duree['fin'] - duree['debut']).dt.total_seconds() / 60


print(f"Durée médiane d'une alerte : {duree['duree_minutes'].median():.2f} minutes")
print(f"Durée moyenne              : {duree['duree_minutes'].mean():.2f} minutes")
print(f"Durée min                  : {duree['duree_minutes'].min():.2f} minutes")
print(f"Durée max                  : {duree['duree_minutes'].max():.2f} minutes")

Durée médiane d'une alerte : 8.92 minutes
Durée moyenne              : 29.90 minutes
Durée min                  : 0.00 minutes
Durée max                  : 578.72 minutes


In [ ]:
# Nombre d'alertes orageuses

nb_alertes = df.groupby(['airport', 'airport_alert_id']).ngroups
print(f"Nombre d'alertes orageuses : {nb_alertes}")

Nombre d'alertes orageuses : 2627
